<a href="https://colab.research.google.com/github/Sneha-Tripathi/infosys-maritime-ai/blob/Milestone1/Milestone1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q streamlit streamlit-option-menu bcrypt pyngrok PyJWT pandas


In [2]:
%%writefile app.py
import os
import re
import html
import sqlite3
import secrets
import smtplib
import datetime as dt
import time
from email.message import EmailMessage

import bcrypt
import jwt
import pandas as pd
import streamlit as st
from streamlit_option_menu import option_menu


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
DB_PATH = "infosys_portal.db"
JWT_SECRET = os.getenv("JWT_SECRET", "")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")
ADMIN_EMAIL = os.getenv("ADMIN_EMAIL", "").strip().lower()
OTP_EXPIRY_SECONDS = 300
LOCK_DURATION_SECONDS = 10 * 60
MAX_LOGIN_FAILURES = 6
TEMP_LOCK_FAILURES = 3
MAX_OTP_FAILURES = 3

SECURITY_QUESTIONS = [
    "What is your pet name?",
    "Who is your best friend?",
    "What is your mother's maiden name?",
    "What is your favourite city?",
]

st.set_page_config(
    page_title="FreightQuote AI",
    page_icon="F",
    layout="wide",
    initial_sidebar_state="expanded",
)


# -----------------------------------------------------------------------------
# UI theme
# -----------------------------------------------------------------------------
st.markdown(
    """
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

    :root {
        --background: #F4F9FF;
        --surface: #FFFFFF;
        --primary: #3478F6;
        --primary-dark: #2458B8;
        --accent: #FFD966;
        --text: #263858;
        --muted: #70809A;
        --border: #DCE8F5;
        --soft-blue: #E8F3FF;
        --soft-yellow: #FFF8DB;
    }

    html, body, [class*="css"] { font-family: 'Inter', sans-serif; }
    .stApp, [data-testid="stAppViewContainer"] { background: var(--background); color: var(--text); }
    [data-testid="stHeader"] { background: rgba(244,249,255,.95); }
    .main .block-container { max-width: 1200px; padding-top: 1.7rem; padding-bottom: 3rem; }
    [data-testid="stSidebar"] {
        background: linear-gradient(180deg, #E7F3FF 0%, #F2F9FF 65%, #FFFBEA 100%);
        border-right: 1px solid var(--border);
    }
    [data-testid="stSidebar"] * { color: var(--text); }
    #MainMenu, footer, header { visibility: hidden; }

    h1, h2, h3 { color: #203B69 !important; font-weight: 800 !important; }
    p, label, span, div { font-family: 'Inter', sans-serif; }
    .brand-name { color: #2458B8; font-size: 2.05rem; font-weight: 800; letter-spacing: -.8px; margin-bottom: 3px; }
    .brand-tagline, .subtitle { color: var(--muted); font-size: .95rem; line-height: 1.7; }
    .custom-card, .auth-card, .metric-card {
        background: var(--surface); border: 1px solid var(--border); border-radius: 18px;
        padding: 22px; box-shadow: 0 8px 28px rgba(48,91,145,.06); margin-bottom: 16px;
    }
    .metric-card { min-height: 112px; }
    .metric-label { color: var(--muted); font-size: .88rem; margin-bottom: 8px; }
    .metric-value { color: var(--primary-dark); font-size: 1.7rem; font-weight: 800; }
    .fq-header {
        text-align: center; padding: 30px 18px; margin-bottom: 26px;
        background: linear-gradient(120deg, #DCEEFF 0%, #FFFFFF 58%, #FFF5CC 100%);
        border: 1px solid #C9DFF5; border-radius: 20px;
        box-shadow: 0 10px 28px rgba(36, 88, 184, .08);
    }
    [data-testid="stTextInput"] input:focus,
    [data-testid="stPasswordInput"] input:focus {
        border-color: #3478F6 !important;
        box-shadow: 0 0 0 2px rgba(52,120,246,.14) !important;
    }
    [data-testid="stForm"] { border-color: #DCE8F5; }
    .welcome-card {
        background: linear-gradient(120deg, #E5F2FF 0%, #FFFFFF 65%, #FFF8D9 100%);
        border: 1px solid var(--border); border-radius: 18px; padding: 24px; margin-bottom: 20px;
    }

    /* Streamlit DOM differs between releases; target both primary-button attributes and form wrappers. */
    .stButton > button,
    .stFormSubmitButton > button,
    [data-testid="stFormSubmitButton"],
    [data-testid="stFormSubmitButton"] button,
    button[data-testid="stBaseButton-primary"],
    [data-testid="stBaseButton-primary"] {
        border-radius: 11px !important;
        min-height: 44px !important;
        font-weight: 650 !important;
        transition: background-color .18s ease, border-color .18s ease, box-shadow .18s ease !important;
    }
    /* Secondary navigation buttons are outlined. */
    .stButton > button:not([kind="primary"]):not([data-testid="stBaseButton-primary"]) {
        background: #FFFFFF !important;
        background-color: #FFFFFF !important;
        color: #2458B8 !important;
        border: 1px solid #C8DDF5 !important;
        box-shadow: none !important;
    }
    .stButton > button:not([kind="primary"]):not([data-testid="stBaseButton-primary"]):hover {
        background: #EAF3FF !important;
        border-color: #3478F6 !important;
        color: #2458B8 !important;
    }

    /* Default button color: Dark blue */
    .stButton > button,
    .stFormSubmitButton > button,
    [data-testid="stFormSubmitButton"],
    [data-testid="stFormSubmitButton"] button,
    button[data-testid="stBaseButton-primary"],
    [data-testid="stBaseButton-primary"],
    button[kind="primary"],
    button[kind="secondary"],
    form button {
        background: #2458B8 !important;
        background-color: #2458B8 !important;
        color: #FFFFFF !important;
        border: 1px solid #2458B8 !important;
        border-radius: 11px !important;
        min-height: 44px !important;
        font-weight: 600 !important;
        box-shadow: none !important;
        transition: background-color 0.2s ease,
                     border-color 0.2s ease !important;
  }

   /* On hover: Bright blue */
   .stButton > button:hover,
   .stFormSubmitButton > button:hover,
   [data-testid="stFormSubmitButton"]:hover,
   [data-testid="stFormSubmitButton"] button:hover,
   button[data-testid="stBaseButton-primary"]:hover,
   [data-testid="stBaseButton-primary"]:hover,
   button[kind="primary"]:hover,
   button[kind="secondary"]:hover,
   form button:hover {
       background: #3478F6 !important;
       background-color: #3478F6 !important;
       color: #FFFFFF !important;
       border-color: #3478F6 !important;
       box-shadow: 0 4px 12px rgba(52, 120, 246, 0.2) !important;
   }

    .auth-layout { gap: 2rem; align-items: stretch; }
    .auth-hero {
        min-height: 480px; padding: 34px; border-radius: 24px;
        background: linear-gradient(145deg, #DCEEFF 0%, #FFFFFF 58%, #FFF5CC 100%);
        border: 1px solid #C9DFF5; box-shadow: 0 16px 40px rgba(36,88,184,.09);
        display: flex; flex-direction: column; justify-content: center;
    }
    .auth-eyebrow { color: #3478F6; font-size: 11px; font-weight: 800; letter-spacing: 1.7px; margin-bottom: 16px; }
    .auth-hero-title { color: #15366D; font-size: clamp(30px,3.2vw,43px); font-weight: 800; line-height: 1.12; letter-spacing: -1.1px; margin: 0 0 16px; }
    .auth-hero-copy { color: #607797; font-size: 15px; line-height: 1.85; margin: 0 0 24px; }
    .auth-feature { color: #314B70; font-size: 14px; line-height: 1.6; padding: 11px 13px; margin: 9px 0; background: rgba(255,255,255,.72); border: 1px solid rgba(201,223,245,.9); border-radius: 12px; }
    .auth-feature strong { color: #2458B8; }
    .auth-form-heading { margin: 4px 0 5px; font-size: 28px; font-weight: 800; color: #203B69; }
    .auth-form-subtitle { color: #70809A; line-height: 1.7; margin-bottom: 22px; }
    @media (max-width: 850px) { .auth-hero { min-height: auto; padding: 24px; margin-bottom: 10px; } }
    .stTextInput input, .stPasswordInput input, [data-baseweb="select"] { border-radius: 10px !important; }
    [data-testid="stAlert"] { border-radius: 12px; }
    hr { border-color: var(--border); }
    a { color: var(--primary-dark); }
    [data-testid="stProgress"] > div > div { background-color: var(--primary); }
    [data-testid="stDataFrame"] { border: 1px solid var(--border); border-radius: 12px; overflow: hidden; }
    </style>
    """,
    unsafe_allow_html=True,
)


# -----------------------------------------------------------------------------
# Database setup and schema migration
# -----------------------------------------------------------------------------
def get_connection():
    connection = sqlite3.connect(DB_PATH, check_same_thread=False, timeout=15)
    connection.row_factory = sqlite3.Row
    return connection


def init_db():
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute(
            """
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE COLLATE NOCASE NOT NULL,
                email TEXT UNIQUE COLLATE NOCASE NOT NULL,
                password_hash BLOB NOT NULL,
                security_question TEXT,
                security_answer_hash BLOB,
                role TEXT NOT NULL DEFAULT 'user',
                created_at TEXT,
                failed_attempts INTEGER NOT NULL DEFAULT 0,
                locked_until REAL NOT NULL DEFAULT 0,
                permanently_locked INTEGER NOT NULL DEFAULT 0,
                otp_failed_attempts INTEGER NOT NULL DEFAULT 0,
                otp_locked_until REAL NOT NULL DEFAULT 0,
                security_failed_attempts INTEGER NOT NULL DEFAULT 0
            )
            """
        )
        cursor.execute("PRAGMA table_info(users)")
        existing_columns = {row["name"] for row in cursor.fetchall()}
        migrations = {
            "role": "ALTER TABLE users ADD COLUMN role TEXT NOT NULL DEFAULT 'user'",
            "created_at": "ALTER TABLE users ADD COLUMN created_at TEXT",
            "failed_attempts": "ALTER TABLE users ADD COLUMN failed_attempts INTEGER NOT NULL DEFAULT 0",
            "locked_until": "ALTER TABLE users ADD COLUMN locked_until REAL NOT NULL DEFAULT 0",
            "permanently_locked": "ALTER TABLE users ADD COLUMN permanently_locked INTEGER NOT NULL DEFAULT 0",
            "otp_failed_attempts": "ALTER TABLE users ADD COLUMN otp_failed_attempts INTEGER NOT NULL DEFAULT 0",
            "otp_locked_until": "ALTER TABLE users ADD COLUMN otp_locked_until REAL NOT NULL DEFAULT 0",
            "security_failed_attempts": "ALTER TABLE users ADD COLUMN security_failed_attempts INTEGER NOT NULL DEFAULT 0",
        }
        for column, statement in migrations.items():
            if column not in existing_columns:
                cursor.execute(statement)

        now_iso = dt.datetime.now(dt.timezone.utc).isoformat()
        cursor.execute(
            "UPDATE users SET created_at = ? WHERE created_at IS NULL OR created_at = ''",
            (now_iso,),
        )

        # Keep administrator access limited to the configured administrator email.
        if ADMIN_EMAIL:
            cursor.execute(
                "UPDATE users SET role = 'user' WHERE lower(email) != ? AND lower(role) = 'admin'",
                (ADMIN_EMAIL,),
            )
            cursor.execute("UPDATE users SET role = 'admin' WHERE lower(email) = ?", (ADMIN_EMAIL,))

        conn.commit()
    finally:
        conn.close()


init_db()


# -----------------------------------------------------------------------------
# Validation, hashing and authentication helpers
# -----------------------------------------------------------------------------
def valid_email(email):
    email = email.strip()
    pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
    return bool(re.fullmatch(pattern, email))


def password_error(password):
    if len(password) < 8:
        return "Password must contain at least 8 characters."
    if not re.search(r"[A-Z]", password):
        return "Password must contain at least one uppercase letter."
    if not re.search(r"[a-z]", password):
        return "Password must contain at least one lowercase letter."
    if not re.search(r"\d", password):
        return "Password must contain at least one digit."
    if not re.search(r"[^A-Za-z0-9]", password):
        return "Password must contain at least one special character."
    return None


def hash_password(password):
    return bcrypt.hashpw(password.encode("utf-8"), bcrypt.gensalt())


def verify_password(password, stored_hash):
    if not password or not stored_hash:
        return False
    try:
        if isinstance(stored_hash, str):
            stored_hash = stored_hash.encode("utf-8")
        return bcrypt.checkpw(password.encode("utf-8"), stored_hash)
    except (ValueError, TypeError):
        return False


def security_answer_error(answer):
    if not answer:
        return "Please enter your security answer."
    if any(character.isspace() for character in answer):
        return "Security answer must not contain spaces or other whitespace characters."
    return None


def hash_security_answer(answer):
    """Hash the exact answer. No lowercasing or trimming: answers are case-sensitive."""
    error = security_answer_error(answer)
    if error:
        raise ValueError(error)
    return bcrypt.hashpw(answer.encode("utf-8"), bcrypt.gensalt())


def verify_security_answer(answer, stored_hash):
    """Verify the exact case-sensitive answer; whitespace is not permitted."""
    if security_answer_error(answer) or not stored_hash:
        return False
    try:
        if isinstance(stored_hash, str):
            stored_hash = stored_hash.encode("utf-8")
        return bcrypt.checkpw(answer.encode("utf-8"), stored_hash)
    except (ValueError, TypeError):
        return False


def make_jwt(email):
    if not JWT_SECRET:
        raise RuntimeError("JWT_SECRET is not configured. Set the JWT_SECRET environment variable before login.")
    now = dt.datetime.now(dt.timezone.utc)
    payload = {
        "email": email.strip().lower(),
        "exp": now + dt.timedelta(hours=2),
        "iat": now,
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


def verify_jwt(token):
    if not token or not JWT_SECRET:
        return None
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        return None


def get_user_by_email(email):
    conn = get_connection()
    try:
        return conn.execute(
            "SELECT * FROM users WHERE lower(email) = ?",
            (email.strip().lower(),),
        ).fetchone()
    finally:
        conn.close()


def get_user_by_username_or_email(value):
    value = value.strip().lower()
    conn = get_connection()
    try:
        return conn.execute(
            "SELECT * FROM users WHERE lower(email) = ? OR lower(username) = ?",
            (value, value),
        ).fetchone()
    finally:
        conn.close()


def get_lock_remaining(timestamp):
    return max(0, int(float(timestamp or 0) - time.time() + 0.999))


def format_duration(seconds):
    seconds = max(0, int(seconds))
    minutes, remainder = divmod(seconds, 60)
    if minutes:
        return f"{minutes} minute(s) and {remainder} second(s)"
    return f"{remainder} second(s)"


def record_login_failure(user_id):
    """Persist a failed login and apply temporary/permanent account locks."""
    conn = get_connection()
    try:
        conn.execute("BEGIN IMMEDIATE")
        user = conn.execute(
            "SELECT failed_attempts, locked_until, permanently_locked FROM users WHERE id = ?",
            (user_id,),
        ).fetchone()
        if not user:
            conn.commit()
            return "invalid", 0, 0
        if user["permanently_locked"]:
            conn.commit()
            return "permanent", 0, int(user["failed_attempts"])
        now = time.time()
        remaining = get_lock_remaining(user["locked_until"])
        if remaining > 0:
            conn.commit()
            return "temporary", remaining, int(user["failed_attempts"])

        failures = int(user["failed_attempts"] or 0) + 1
        if failures >= MAX_LOGIN_FAILURES:
            conn.execute(
                "UPDATE users SET failed_attempts = ?, permanently_locked = 1, locked_until = 0 WHERE id = ?",
                (failures, user_id),
            )
            conn.commit()
            return "permanent", 0, failures

        if failures == TEMP_LOCK_FAILURES:
            locked_until = now + LOCK_DURATION_SECONDS
            conn.execute(
                "UPDATE users SET failed_attempts = ?, locked_until = ? WHERE id = ?",
                (failures, locked_until, user_id),
            )
            conn.commit()
            return "temporary", LOCK_DURATION_SECONDS, failures

        conn.execute(
            "UPDATE users SET failed_attempts = ?, locked_until = 0 WHERE id = ?",
            (failures, user_id),
        )
        conn.commit()
        return "failed", 0, failures
    finally:
        conn.close()


def reset_login_failures_after_success(user_id):
    conn = get_connection()
    try:
        conn.execute(
            "UPDATE users SET failed_attempts = 0, locked_until = 0 WHERE id = ? AND permanently_locked = 0",
            (user_id,),
        )
        conn.commit()
    finally:
        conn.close()


def record_security_answer_failure(user_id):
    """Count an incorrect security answer and temporarily lock the account after 3 failures."""
    conn = get_connection()
    try:
        conn.execute("BEGIN IMMEDIATE")
        user = conn.execute(
            "SELECT security_failed_attempts, locked_until, permanently_locked FROM users WHERE id = ?",
            (user_id,),
        ).fetchone()
        if not user:
            conn.commit()
            return "invalid", 0, 0
        if user["permanently_locked"]:
            conn.commit()
            return "permanent", 0, int(user["security_failed_attempts"] or 0)

        now = time.time()
        remaining = get_lock_remaining(user["locked_until"])
        if remaining > 0:
            conn.commit()
            return "locked", remaining, int(user["security_failed_attempts"] or 0)

        # If the previous temporary lock has expired, begin a fresh security-answer window.
        previous_locked_until = float(user["locked_until"] or 0)
        failures_before = int(user["security_failed_attempts"] or 0)
        if previous_locked_until > 0 and previous_locked_until <= now:
            failures_before = 0

        failures = failures_before + 1
        if failures >= 3:
            locked_until = now + LOCK_DURATION_SECONDS
            conn.execute(
                "UPDATE users SET security_failed_attempts = ?, locked_until = ? WHERE id = ?",
                (failures, locked_until, user_id),
            )
            conn.commit()
            return "locked", LOCK_DURATION_SECONDS, failures

        conn.execute(
            "UPDATE users SET security_failed_attempts = ?, locked_until = 0 WHERE id = ?",
            (failures, user_id),
        )
        conn.commit()
        return "failed", 0, failures
    finally:
        conn.close()


def reset_security_answer_failures(user_id):
    """Clear wrong security-answer attempts after a successful answer."""
    conn = get_connection()
    try:
        conn.execute(
            "UPDATE users SET security_failed_attempts = 0 WHERE id = ? AND permanently_locked = 0",
            (user_id,),
        )
        conn.commit()
    finally:
        conn.close()


def get_otp_lock_remaining(email):
    """Return OTP lock time, clearing an expired OTP lock when appropriate."""
    conn = get_connection()
    try:
        user = conn.execute(
            "SELECT otp_locked_until FROM users WHERE lower(email) = ?",
            (email.strip().lower(),),
        ).fetchone()
        if not user:
            return 0
        remaining = get_lock_remaining(user["otp_locked_until"])
        if remaining == 0 and float(user["otp_locked_until"] or 0) > 0:
            conn.execute(
                "UPDATE users SET otp_failed_attempts = 0, otp_locked_until = 0 WHERE lower(email) = ?",
                (email.strip().lower(),),
            )
            conn.commit()
        return remaining
    finally:
        conn.close()


def record_otp_failure(email):
    conn = get_connection()
    try:
        conn.execute("BEGIN IMMEDIATE")
        user = conn.execute(
            "SELECT otp_failed_attempts, otp_locked_until FROM users WHERE lower(email) = ?",
            (email.strip().lower(),),
        ).fetchone()
        if not user:
            conn.commit()
            return "invalid", 0, 0

        remaining = get_lock_remaining(user["otp_locked_until"])
        if remaining > 0:
            conn.commit()
            return "locked", remaining, int(user["otp_failed_attempts"])

        # Start a fresh OTP attempt counter when a previous lock has expired.
        previous_lock = float(user["otp_locked_until"] or 0)
        failures_before = int(user["otp_failed_attempts"] or 0)
        if previous_lock > 0 and previous_lock <= time.time():
            failures_before = 0

        failures = failures_before + 1
        if failures >= MAX_OTP_FAILURES:
            locked_until = time.time() + LOCK_DURATION_SECONDS
            conn.execute(
                "UPDATE users SET otp_failed_attempts = ?, otp_locked_until = ? WHERE lower(email) = ?",
                (failures, locked_until, email.strip().lower()),
            )
            conn.commit()
            return "locked", LOCK_DURATION_SECONDS, failures

        conn.execute(
            "UPDATE users SET otp_failed_attempts = ?, otp_locked_until = 0 WHERE lower(email) = ?",
            (failures, email.strip().lower()),
        )
        conn.commit()
        return "failed", 0, failures
    finally:
        conn.close()


def reset_otp_failures(email):
    conn = get_connection()
    try:
        conn.execute(
            "UPDATE users SET otp_failed_attempts = 0, otp_locked_until = 0 WHERE lower(email) = ?",
            (email.strip().lower(),),
        )
        conn.commit()
    finally:
        conn.close()


def unlock_account(user_id):
    """Admin-only interface calls this to remove a permanent account lock."""
    conn = get_connection()
    try:
        conn.execute(
            """
            UPDATE users
            SET failed_attempts = 0,
                locked_until = 0,
                permanently_locked = 0,
                otp_failed_attempts = 0,
                otp_locked_until = 0,
                security_failed_attempts = 0
            WHERE id = ?
            """,
            (user_id,),
        )
        conn.commit()
    finally:
        conn.close()


# -----------------------------------------------------------------------------
# Email OTP
# -----------------------------------------------------------------------------
def send_otp_email(recipient_email, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        return False, "Email configuration is missing. Please configure EMAIL_ADDRESS and EMAIL_PASSWORD."

    message = EmailMessage()
    message["Subject"] = "Your FreightQuote AI password reset code"
    message["From"] = EMAIL_ADDRESS
    message["To"] = recipient_email

    text_body = (
        "Hello,\n\n"
        "We received a request to reset the password for your FreightQuote AI account.\n\n"
        f"Your verification code is: {otp}\n\n"
        "This code is valid for 5 minutes and can be used only for this password-reset request.\n"
        "If you did not request this change, you can safely ignore this email.\n\n"
        "Regards,\nFreightQuote AI Security Team"
    )
    message.set_content(text_body)
    message.add_alternative(
        f"""\
        <!doctype html>
        <html>
          <body style="margin:0;padding:0;background:#f4f9ff;font-family:Arial,Helvetica,sans-serif;color:#263858;">
            <div style="max-width:560px;margin:28px auto;background:#ffffff;border:1px solid #dce8f5;border-radius:16px;overflow:hidden;">
              <div style="padding:22px 28px;background:linear-gradient(120deg,#e5f2ff,#ffffff,#fff8d9);border-bottom:1px solid #dce8f5;">
                <div style="font-size:24px;font-weight:700;color:#15366d;">FreightQuote <span style="color:#3478f6;">AI</span></div>
                <div style="margin-top:6px;font-size:13px;color:#70809a;">Account security notification</div>
              </div>
              <div style="padding:28px;">
                <h2 style="margin:0 0 12px;color:#203b69;font-size:21px;">Password reset verification</h2>
                <p style="margin:0 0 18px;line-height:1.7;">We received a request to reset the password for your FreightQuote AI account. Enter the verification code below to continue.</p>
                <div style="padding:18px;background:#f4f9ff;border:1px solid #dce8f5;border-radius:12px;text-align:center;">
                  <div style="font-size:12px;letter-spacing:1.5px;color:#70809a;text-transform:uppercase;">Your verification code</div>
                  <div style="font-size:32px;font-weight:800;letter-spacing:8px;color:#2458b8;padding:10px 0;">{html.escape(otp)}</div>
                  <div style="font-size:13px;color:#70809a;">Valid for 5 minutes</div>
                </div>
                <p style="margin:20px 0 0;line-height:1.7;">If you did not request a password reset, ignore this email. Do not share this code with anyone.</p>
              </div>
              <div style="padding:16px 28px;background:#fbfdff;border-top:1px solid #dce8f5;font-size:12px;color:#70809a;">FreightQuote AI Security Team</div>
            </div>
          </body>
        </html>
        """,
        subtype="html",
    )

    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
            server.starttls()
            server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
            server.send_message(message)
        return True, "OTP sent successfully."
    except Exception:
        return False, "Unable to send the OTP email. Please check your email configuration and try again."


# -----------------------------------------------------------------------------
# Session state and reusable layout helpers
# -----------------------------------------------------------------------------
SESSION_DEFAULTS = {
    "page": "Login",
    "token": None,
    "reset_email": "",
    "reset_mode": "",
    "otp_code": None,
    "otp_expires": 0,
    "otp_verified": False,
    "sq_email": "",
    "sq_question": "",
    "sq_verified": False,
    "admin_logged_in": False,
}
for key, default_value in SESSION_DEFAULTS.items():
    if key not in st.session_state:
        st.session_state[key] = default_value


def navigate(page):
    st.session_state.page = page
    st.rerun()


def clear_reset():
    st.session_state.reset_email = ""
    st.session_state.reset_mode = ""
    st.session_state.otp_code = None
    st.session_state.otp_expires = 0
    st.session_state.otp_verified = False
    st.session_state.sq_email = ""
    st.session_state.sq_question = ""
    st.session_state.sq_verified = False


def header(title, subtitle="AI-powered Freight Quotation Platform"):
    st.markdown(
        f"""
        <div style="margin-bottom:24px;">
            <div class="brand-name">FreightQuote <span style="color:#3478F6;">AI</span></div>
            <div class="brand-tagline">{html.escape(subtitle)}</div>
            <h1 style="font-size:2rem;margin:14px 0 6px;">{html.escape(title)}</h1>
        </div>
        """,
        unsafe_allow_html=True,
    )


def login_user(email):
    if not JWT_SECRET:
        raise RuntimeError("JWT_SECRET is not configured. Set the JWT_SECRET environment variable before login.")
    st.session_state.token = make_jwt(email)
    st.session_state.admin_logged_in = False
    st.session_state.page = "Dashboard"
    st.rerun()


def render_auth_banner():
    st.markdown(
        """
        <div class="welcome-card">
          <div style="font-size:12px;font-weight:700;color:#3478F6;letter-spacing:1.4px;margin-bottom:10px;">SMARTER FREIGHT MANAGEMENT</div>
          <h2 style="font-size:26px;margin:0 0 10px;">Freight decisions, made simpler.</h2>
          <p style="color:#70809A;font-size:14px;line-height:1.8;margin:0;">Access your account to manage freight quotations and explore your FreightQuote AI workspace.</p>
        </div>
        """,
        unsafe_allow_html=True,
    )


# -----------------------------------------------------------------------------
# Login page
# -----------------------------------------------------------------------------
def render_login_page():
    """Render the updated split-layout login page."""
    left_col, right_col = st.columns([1.05, 0.95], gap="large")

    with left_col:
        st.markdown(
            """
            <div class="auth-hero">
              <div class="auth-eyebrow">FREIGHTQUOTE AI · SMARTER FREIGHT MANAGEMENT</div>
              <div class="auth-hero-title">Freight decisions,<br><span style="color:#3478F6;">made simpler.</span></div>
              <p class="auth-hero-copy">A secure workspace for accessing your FreightQuote AI account and managing your freight-related activities.</p>
              <div class="auth-feature"><strong>Secure access</strong><br>Protected account login and token-based sessions.</div>
              <div class="auth-feature"><strong>Account recovery</strong><br>Reset your password using security verification or email OTP.</div>
              <div class="auth-feature"><strong>One workspace</strong><br>Access your dashboard and available account analytics.</div>
              <div style="margin-top:24px;color:#70809A;font-size:12px;">FreightQuote AI · Account Access Portal</div>
            </div>
            """,
            unsafe_allow_html=True,
        )

    with right_col:
        st.markdown(
            """
            <div style="padding:8px 3px 18px;">
              <div style="font-size:13px;font-weight:800;letter-spacing:1.2px;color:#3478F6;text-transform:uppercase;margin-bottom:8px;">Welcome back</div>
              <div class="auth-form-heading">Sign in to your account</div>
              <div class="auth-form-subtitle">Enter your account details to continue to your workspace.</div>
            </div>
            """,
            unsafe_allow_html=True,
        )

        with st.container(border=True):
            with st.form("login_form"):
                login_identity = st.text_input(
                    "Username or Email",
                    placeholder="Enter your username or email",
                )
                login_password = st.text_input(
                    "Password",
                    type="password",
                    placeholder="Enter your password",
                )
                st.checkbox("Remember me", key="remember_me")
                submitted = st.form_submit_button(
                    "Sign In",
                    type="primary",
                    use_container_width=True,
                )

            if submitted:
                if not login_identity.strip() or not login_password:
                    st.error("Please enter your username/email and password.")
                else:
                    user = get_user_by_username_or_email(login_identity)
                    if not user:
                        st.error("Invalid username/email or password.")
                    elif user["permanently_locked"]:
                        st.error("This account is permanently locked after six failed login attempts. Contact an administrator to unlock it.")
                    else:
                        remaining = get_lock_remaining(user["locked_until"])
                        if remaining > 0:
                            st.error(f"Account temporarily locked. Try again in {format_duration(remaining)}.")
                        elif verify_password(login_password, user["password_hash"]):
                            reset_login_failures_after_success(user["id"])
                            try:
                                login_user(user["email"])
                            except RuntimeError as exc:
                                st.error(str(exc))
                        else:
                            status, remaining, attempts = record_login_failure(user["id"])
                            if status == "permanent":
                                st.error("This account is now permanently locked after six failed login attempts. Only an administrator can unlock it.")
                            elif status == "temporary":
                                st.error(f"Too many failed login attempts. Your account is locked for 10 minutes. Try again in {format_duration(remaining)}.")
                            else:
                                attempts_left = max(0, MAX_LOGIN_FAILURES - attempts)
                                st.error(f"Invalid username/email or password. Failed attempts: {attempts}/{MAX_LOGIN_FAILURES}. {attempts_left} attempt(s) remain before permanent lock.")

            st.markdown(
                "<p style='text-align:center;color:#70809A;margin:15px 0 10px;'>New to FreightQuote AI?</p>",
                unsafe_allow_html=True,
            )
            action_left, action_right = st.columns(2, gap="small")
            with action_left:
                if st.button("Create Account", use_container_width=True):
                    navigate("Signup")
            with action_right:
                if st.button("Forgot Password?", use_container_width=True):
                    clear_reset()
                    navigate("Forgot")
            st.markdown(
                """
                <div style="border-top:1px solid #DCE8F5;padding-top:15px;margin-top:15px;color:#70809A;font-size:12px;line-height:1.7;">
                  <b style="color:#2458B8;">Account security</b><br>
                  Password hashing, account lockout controls, and token-based authentication help protect your account.
                </div>
                """,
                unsafe_allow_html=True,
            )


def render_signup_page():
    """Render the updated split-layout account registration page."""
    left_col, right_col = st.columns([1.0, 1.15], gap="large")

    with left_col:
        st.markdown(
            """
            <div class="auth-hero">
              <div class="auth-eyebrow">CREATE YOUR WORKSPACE</div>
              <div class="auth-hero-title">Start with a<br><span style="color:#3478F6;">secure account.</span></div>
              <p class="auth-hero-copy">Create your FreightQuote AI account to access your workspace. Choose a strong password and set up your account recovery details carefully.</p>
              <div class="auth-feature"><strong>Strong passwords</strong><br>Use at least 8 characters with uppercase, lowercase, a number and a special character.</div>
              <div class="auth-feature"><strong>Recovery protection</strong><br>Your security answer is case-sensitive and must not contain spaces.</div>
              <div class="auth-feature"><strong>Role-based access</strong><br>Your available workspace depends on your assigned account role.</div>
            </div>
            """,
            unsafe_allow_html=True,
        )

    with right_col:
        st.markdown(
            """
            <div style="padding:8px 3px 16px;">
              <div style="font-size:13px;font-weight:800;letter-spacing:1.2px;color:#3478F6;text-transform:uppercase;margin-bottom:8px;">New account</div>
              <div class="auth-form-heading">Create your account</div>
              <div class="auth-form-subtitle">Fill in the details below to set up your FreightQuote AI access.</div>
            </div>
            """,
            unsafe_allow_html=True,
        )

        with st.container(border=True):
            with st.form("signup_form"):
                username = st.text_input("Username", placeholder="Choose a username")
                email = st.text_input("Email Address", placeholder="example@gmail.com")
                password = st.text_input("Password", type="password", placeholder="Create a strong password")
                confirm_password = st.text_input("Confirm Password", type="password", placeholder="Re-enter your password")
                role_choice = st.selectbox(
                    "Account Role",
                    ["User", "Admin"],
                    help="Administrator access is restricted to the email configured in ADMIN_EMAIL.",
                )
                st.markdown("#### Account Recovery")
                security_question = st.selectbox("Security Question", SECURITY_QUESTIONS)
                security_answer = st.text_input(
                    "Security Answer",
                    placeholder="Enter your exact answer without spaces",
                    help="The answer is case-sensitive. Spaces are not allowed.",
                )
                create_account = st.form_submit_button(
                    "Create Account",
                    type="primary",
                    use_container_width=True,
                )

            if create_account:
                username = username.strip()
                email = email.strip().lower()
                answer_error = security_answer_error(security_answer)

                if not all([username, email, password, confirm_password, security_answer]):
                    st.error("Please fill in all the required fields.")
                elif not valid_email(email):
                    st.error("Please enter a valid email address.")
                elif len(username) < 3:
                    st.error("Username must contain at least 3 characters.")
                elif len(username) > 50:
                    st.error("Username cannot exceed 50 characters.")
                elif any(character.isspace() for character in username):
                    st.error("Username cannot contain spaces.")
                elif not re.fullmatch(r"[A-Za-z0-9_.-]+", username):
                    st.error("Username may contain letters, digits, underscores, dots and hyphens only.")
                elif password_error(password):
                    st.error(password_error(password))
                elif password != confirm_password:
                    st.error("Passwords do not match.")
                elif answer_error:
                    st.error(answer_error)
                elif role_choice == "Admin" and (not ADMIN_EMAIL or email != ADMIN_EMAIL):
                    st.error("Administrator access is restricted. Use the email configured in the ADMIN_EMAIL secret or select User.")
                else:
                    assigned_role = "admin" if (role_choice == "Admin" or (ADMIN_EMAIL and email == ADMIN_EMAIL)) else "user"
                    conn = get_connection()
                    try:
                        conn.execute(
                            """
                            INSERT INTO users (
                                username, email, password_hash, security_question,
                                security_answer_hash, role, created_at
                            ) VALUES (?, ?, ?, ?, ?, ?, ?)
                            """,
                            (
                                username,
                                email,
                                hash_password(password),
                                security_question,
                                hash_security_answer(security_answer),
                                assigned_role,
                                dt.datetime.now(dt.timezone.utc).isoformat(),
                            ),
                        )
                        conn.commit()
                        st.success(f"Account created successfully with the {assigned_role.title()} role. Please sign in.")
                        st.session_state.page = "Login"
                        st.rerun()
                    except sqlite3.IntegrityError:
                        st.error("This username or email is already registered.")
                    except (sqlite3.Error, ValueError):
                        st.error("Unable to create your account. Please check the details and try again.")
                    finally:
                        conn.close()

            st.markdown("<div style='margin-top:12px;color:#70809A;'>Already have an account?</div>", unsafe_allow_html=True)
            if st.button("Back to Sign In", use_container_width=True):
                navigate("Login")


def render_forgot_page():
    header("Reset your password", "Choose a recovery method to regain access to your account.")
    st.markdown("### Account Recovery")
    reset_method = st.radio(
        "Choose a recovery method",
        ["Security Question", "Email OTP"],
        horizontal=True,
        key="reset_method_choice",
    )
    reset_email = st.text_input(
        "Registered Email Address",
        value=st.session_state.reset_email,
        placeholder="Enter your registered email",
        key="forgot_email_input",
    )

    if reset_method == "Security Question":
        if st.button("Continue with Security Question", type="primary", use_container_width=True):
            email = reset_email.strip().lower()
            if not valid_email(email):
                st.error("Please enter a valid registered email address.")
            else:
                user = get_user_by_email(email)
                if not user:
                    st.error("No account was found with this email.")
                elif not user["security_question"]:
                    st.error("No security question is configured for this account.")
                else:
                    st.session_state.sq_email = email
                    st.session_state.sq_question = user["security_question"]
                    st.session_state.sq_verified = False
                    st.session_state.reset_email = email
                    st.session_state.reset_mode = "Security Question"
                    navigate("ForgotSecurity")
    else:
        if st.button("Send Email OTP", type="primary", use_container_width=True):
            email = reset_email.strip().lower()
            if not valid_email(email):
                st.error("Please enter a valid registered email address.")
            else:
                user = get_user_by_email(email)
                if not user:
                    st.error("No account was found with this email.")
                else:
                    otp_lock_remaining = get_otp_lock_remaining(email)
                    if otp_lock_remaining > 0:
                        st.error(f"OTP verification is temporarily locked. Try again in {format_duration(otp_lock_remaining)}.")
                    elif user["permanently_locked"]:
                        st.error("This account is permanently locked. Contact an administrator to unlock it before resetting the password.")
                    else:
                        otp = f"{secrets.randbelow(1_000_000):06d}"
                        success, message = send_otp_email(email, otp)
                        if success:
                            st.session_state.reset_email = email
                            st.session_state.reset_mode = "Email OTP"
                            st.session_state.otp_code = otp
                            st.session_state.otp_expires = time.time() + OTP_EXPIRY_SECONDS
                            st.session_state.otp_verified = False
                            navigate("ForgotOTP")
                        else:
                            st.error(message)

    st.markdown("---")
    if st.button("Back to Sign In", use_container_width=True):
        clear_reset()
        navigate("Login")


# -----------------------------------------------------------------------------
# Security-question password reset
# -----------------------------------------------------------------------------
def render_security_reset_page():
    """Verify the security answer first, then display a separate password-reset form."""
    header(
        "Verify your identity" if not st.session_state.sq_verified else "Set a new password",
        "Answer your security question before you can reset your password."
        if not st.session_state.sq_verified
        else "Your security answer has been verified. Choose a new password.",
    )

    email = st.session_state.sq_email
    question = st.session_state.sq_question
    if not email or not question:
        st.warning("Your recovery session has expired. Please start again.")
        if st.button("Return to Sign In", type="primary"):
            clear_reset()
            navigate("Login")
        return

    user = get_user_by_email(email)
    if not user:
        st.error("Account not found. Please start the recovery process again.")
        if st.button("Return to Password Recovery", type="primary"):
            clear_reset()
            navigate("Forgot")
        return

    if user["permanently_locked"]:
        st.error("This account is permanently locked. Contact an administrator to unlock it.")
        if st.button("Back to Sign In"):
            clear_reset()
            navigate("Login")
        return

    # A third incorrect answer locks the account. The same lock is respected by login.
    remaining = get_lock_remaining(user["locked_until"])
    if remaining > 0:
        st.error(
            "This account is temporarily locked after repeated failed verification attempts. "
            f"Try again in {format_duration(remaining)}."
        )
        if st.button("Back to Sign In"):
            clear_reset()
            navigate("Login")
        return

    # If a security-answer lock expired, reset only its recovery-attempt counter.
    if int(user["security_failed_attempts"] or 0) >= 3 and float(user["locked_until"] or 0) > 0:
        conn = get_connection()
        try:
            conn.execute(
                "UPDATE users SET security_failed_attempts = 0, locked_until = 0 WHERE id = ? AND permanently_locked = 0",
                (user["id"],),
            )
            conn.commit()
        finally:
            conn.close()
        user = get_user_by_email(email)

    st.caption("Registered email")
    st.info(email)
    st.markdown("#### Security Question")
    st.write(question)

    if not st.session_state.sq_verified:
        st.info("Enter the exact answer. It is case-sensitive and must not contain spaces. Three incorrect answers lock the account for 10 minutes.")
        with st.form("security_answer_verification_form"):
            answer = st.text_input(
                "Security Answer",
                placeholder="Enter your exact answer without spaces",
            )
            verify_button = st.form_submit_button(
                "Verify Security Answer",
                type="primary",
                use_container_width=True,
            )

        if verify_button:
            answer_error = security_answer_error(answer)
            if not answer:
                st.error("Please enter your security answer.")
            elif verify_security_answer(answer, user["security_answer_hash"]):
                reset_security_answer_failures(user["id"])
                st.session_state.sq_verified = True
                st.success("Security answer verified. You can now set a new password.")
                st.rerun()
            else:
                status, lock_remaining, attempts = record_security_answer_failure(user["id"])
                if status == "locked":
                    st.error(
                        "Three incorrect security-answer attempts were entered. "
                        "Your account is locked for 10 minutes."
                    )
                elif status == "permanent":
                    st.error("This account is permanently locked. Contact an administrator.")
                else:
                    attempts_left = max(0, 3 - attempts)
                    message = "Incorrect security answer. Answers are case-sensitive and cannot contain spaces. "
                    if answer_error:
                        message = answer_error + " This attempt has been counted. "
                    st.error(message + f"Failed attempts: {attempts}/3. {attempts_left} attempt(s) remain before a 10-minute lock.")
    else:
        st.success("Security answer verified successfully.")
        with st.form("security_new_password_form"):
            new_password = st.text_input(
                "New Password", type="password", placeholder="Enter your new password"
            )
            confirm_password = st.text_input(
                "Confirm New Password", type="password", placeholder="Re-enter your new password"
            )
            reset_button = st.form_submit_button(
                "Update Password", type="primary", use_container_width=True
            )

        if reset_button:
            # Recheck the account and verification state before changing credentials.
            latest_user = get_user_by_email(email)
            if not latest_user:
                st.error("Account not found. Please start the recovery process again.")
            elif latest_user["permanently_locked"]:
                st.error("This account is permanently locked. Contact an administrator to unlock it.")
            elif get_lock_remaining(latest_user["locked_until"]) > 0:
                st.error("This account is temporarily locked. Wait until the lock expires before continuing.")
            elif not st.session_state.sq_verified:
                st.error("Please verify your security answer first.")
            elif password_error(new_password):
                st.error(password_error(new_password))
            elif new_password != confirm_password:
                st.error("Passwords do not match.")
            elif verify_password(new_password, latest_user["password_hash"]):
                st.error("New password must be different from your current password.")
            else:
                conn = get_connection()
                try:
                    cursor = conn.execute(
                        "UPDATE users SET password_hash = ? WHERE lower(email) = ? AND permanently_locked = 0",
                        (hash_password(new_password), email.strip().lower()),
                    )
                    conn.commit()
                    if cursor.rowcount != 1:
                        st.error("Password could not be updated. Please try again or contact an administrator.")
                    else:
                        clear_reset()
                        st.session_state.page = "Login"
                        st.success("Password updated successfully. Please sign in with your new password.")
                        st.rerun()
                finally:
                    conn.close()

    st.markdown("---")
    if st.button("Cancel", use_container_width=True):
        clear_reset()
        navigate("Login")


# -----------------------------------------------------------------------------
# Email OTP verification and password reset
# -----------------------------------------------------------------------------
def render_otp_reset_page():
    header("Verify email OTP", "Enter the verification code sent to your registered email.")
    email = st.session_state.reset_email
    if not email:
        st.warning("No active OTP request was found. Please request a new OTP.")
        if st.button("Return to Password Recovery"):
            clear_reset()
            navigate("Forgot")
        return

    user = get_user_by_email(email)
    if not user:
        st.error("Account not found. Please start the recovery process again.")
        clear_reset()
        return
    if user["permanently_locked"]:
        st.error("This account is permanently locked. Contact an administrator to unlock it.")
        return

    otp_lock_remaining = get_otp_lock_remaining(email)
    if otp_lock_remaining > 0:
        st.error(f"Too many incorrect OTP attempts. OTP verification is locked for {format_duration(otp_lock_remaining)}.")
        st.info("You cannot request or enter another OTP during this lock period.")
        if st.button("Back to Password Recovery", use_container_width=True):
            clear_reset()
            navigate("Forgot")
        return

    if st.session_state.otp_code is None:
        st.warning("No active OTP request was found. Please request a new OTP.")
        if st.button("Request a New OTP"):
            clear_reset()
            navigate("Forgot")
        return

    st.caption("OTP sent to")
    st.info(email)

    if not st.session_state.otp_verified:
        with st.form("otp_verification_form"):
            entered_otp = st.text_input("6-digit verification code", max_chars=6, placeholder="Enter the code from your email")
            verify_button = st.form_submit_button("Verify OTP", type="primary", use_container_width=True)

        if verify_button:
            if time.time() > float(st.session_state.otp_expires or 0):
                st.error("OTP expired. Please request a new one.")
                st.session_state.otp_code = None
                st.session_state.otp_verified = False
            elif not re.fullmatch(r"\d{6}", entered_otp.strip()):
                if not entered_otp.strip():
                    st.error("Please enter the 6-digit OTP sent to your email.")
                else:
                    status, remaining, attempts = record_otp_failure(email)
                    if status == "locked":
                        st.session_state.otp_code = None
                        st.session_state.otp_verified = False
                        st.error("Three incorrect OTP attempts have been made. OTP verification is locked for 10 minutes.")
                        st.rerun()
                    else:
                        attempts_left = max(0, MAX_OTP_FAILURES - attempts)
                        st.error(f"Please enter a valid 6-digit OTP. Failed attempts: {attempts}/3. {attempts_left} attempt(s) remain before a 10-minute OTP lock.")
            elif not secrets.compare_digest(entered_otp.strip(), str(st.session_state.otp_code)):
                status, remaining, attempts = record_otp_failure(email)
                if status == "locked":
                    st.session_state.otp_code = None
                    st.session_state.otp_verified = False
                    st.error("Three incorrect OTP attempts have been made. OTP verification is locked for 10 minutes.")
                    st.rerun()
                else:
                    attempts_left = max(0, MAX_OTP_FAILURES - attempts)
                    st.error(f"Incorrect OTP. {attempts_left} attempt(s) remain before a 10-minute OTP lock.")
            else:
                reset_otp_failures(email)
                st.session_state.otp_verified = True
                st.success("Email verified successfully. You can now reset your password.")
                st.rerun()

    if st.session_state.otp_verified:
        # Refresh current data before allowing a password change.
        user = get_user_by_email(email)
        if not user or user["permanently_locked"]:
            st.error("This account cannot be updated. Contact an administrator if the account is locked.")
            return
        with st.form("otp_new_password_form"):
            new_password = st.text_input("New Password", type="password", placeholder="Enter your new password")
            confirm_password = st.text_input("Confirm New Password", type="password", placeholder="Re-enter your new password")
            reset_button = st.form_submit_button("Update Password", type="primary", use_container_width=True)

        if reset_button:
            if time.time() > float(st.session_state.otp_expires or 0):
                st.error("OTP verification has expired. Please request a new OTP.")
                clear_reset()
                st.session_state.page = "Forgot"
                st.rerun()
            elif password_error(new_password):
                st.error(password_error(new_password))
            elif new_password != confirm_password:
                st.error("Passwords do not match.")
            elif verify_password(new_password, user["password_hash"]):
                st.error("New password must be different from your current password.")
            else:
                conn = get_connection()
                try:
                    conn.execute(
                        "UPDATE users SET password_hash = ? WHERE lower(email) = ? AND permanently_locked = 0",
                        (hash_password(new_password), email.strip().lower()),
                    )
                    conn.commit()
                    clear_reset()
                    st.success("Password updated successfully. Please sign in with your new password.")
                    st.session_state.page = "Login"
                    st.rerun()
                finally:
                    conn.close()

    st.markdown("---")
    if st.button("Cancel", use_container_width=True):
        clear_reset()
        navigate("Login")


# -----------------------------------------------------------------------------
# Dashboard and analytics
# -----------------------------------------------------------------------------
def render_metric_card(label, value, description):
    st.markdown(
        f"""
        <div class="metric-card">
          <div class="metric-label">{html.escape(str(label))}</div>
          <div class="metric-value">{html.escape(str(value))}</div>
          <div style="color:#70809A;font-size:12px;margin-top:6px;">{html.escape(str(description))}</div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_user_dashboard(user):
    header("Dashboard", "Your FreightQuote AI workspace at a glance.")
    username = html.escape(user["username"])
    email = html.escape(user["email"])
    st.markdown(
        f"""
        <div class="welcome-card">
          <h2 style="margin-top:0;margin-bottom:8px;">Welcome, {username}</h2>
          <p style="color:#6380A5;font-size:15px;margin:0;line-height:1.7;">You are signed in to FreightQuote AI. Your workspace is ready to use.</p>
        </div>
        """,
        unsafe_allow_html=True,
    )

    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric("Workspace", "Active")
    with col2:
        st.metric("Account", "Verified")
    with col3:
        st.metric("Platform", "FreightQuote AI")
    with col4:
        st.metric("Access", "User")

    st.markdown("## Workspace Overview")
    with st.container(border=True):
        st.markdown("### Get started")
        st.write("Your account and authentication are set up. Freight quotation and shipment analytics can be added as those platform modules become available.")

    st.markdown("## Account Details")
    col1, col2 = st.columns(2)
    with col1:
        with st.container(border=True):
            st.caption("USERNAME")
            st.markdown(f"### {username}")
    with col2:
        with st.container(border=True):
            st.caption("REGISTERED EMAIL")
            st.markdown(f"**{email}**")


def render_admin_dashboard(user):
    header("Admin Dashboard", "Manage and monitor registered platform accounts.")
    st.markdown(
        """
        <div class="welcome-card">
          <h2 style="margin-top:0;margin-bottom:8px;">Administrator workspace</h2>
          <p style="color:#6380A5;font-size:14px;margin:0;line-height:1.7;">Review account statistics and manage permanently locked accounts.</p>
        </div>
        """,
        unsafe_allow_html=True,
    )

    conn = get_connection()
    try:
        total_users = conn.execute("SELECT COUNT(*) FROM users").fetchone()[0]
        total_admins = conn.execute("SELECT COUNT(*) FROM users WHERE lower(role) = 'admin'").fetchone()[0]
        users = conn.execute(
            """
            SELECT id, username, email, role, created_at, failed_attempts,
                   locked_until, permanently_locked, otp_locked_until
            FROM users ORDER BY id DESC
            """
        ).fetchall()
    finally:
        conn.close()

    col1, col2, col3 = st.columns(3)
    with col1:
        render_metric_card("Total Accounts", total_users, "All registered accounts")
    with col2:
        render_metric_card("Administrators", total_admins, "Accounts with admin access")
    with col3:
        render_metric_card("Regular Users", total_users - total_admins, "Standard platform accounts")

    st.markdown("### Registered Users")
    if users:
        table_data = []
        for row in users:
            if row["permanently_locked"]:
                account_status = "Permanently locked"
            elif get_lock_remaining(row["locked_until"]) > 0:
                account_status = "Temporarily locked"
            else:
                account_status = "Active"
            table_data.append(
                {
                    "ID": row["id"],
                    "Username": row["username"],
                    "Email": row["email"],
                    "Role": row["role"],
                    "Status": account_status,
                    "Created At": row["created_at"],
                }
            )
        st.dataframe(table_data, use_container_width=True, hide_index=True)
    else:
        st.info("No registered users were found.")

    permanently_locked_users = [row for row in users if row["permanently_locked"]]
    st.markdown("### Permanently Locked Accounts")
    if not permanently_locked_users:
        st.success("There are no permanently locked accounts.")
    else:
        st.warning("Only an administrator can unlock these accounts.")
        for row in permanently_locked_users:
            col1, col2, col3 = st.columns([2, 3, 1.3])
            with col1:
                st.write(f"**{row['username']}**")
            with col2:
                st.caption(row["email"])
            with col3:
                if st.button("Unlock account", key=f"unlock_account_{row['id']}", type="primary", use_container_width=True):
                    unlock_account(row["id"])
                    st.success(f"Account {row['username']} has been unlocked.")
                    st.rerun()


def render_analytics(user):
    is_admin = str(user["role"]).strip().lower() == "admin"
    header("Analytics", "Account and platform activity overview.")

    if not is_admin:
        st.markdown(
            """
            <div class="welcome-card">
              <h2 style="margin-top:0;">Your account overview</h2>
              <p style="color:#6380A5;line-height:1.7;margin-bottom:0;">This page shows information currently available for your account. Freight volume, quotation performance and shipment trends will appear when those modules and their data are added.</p>
            </div>
            """,
            unsafe_allow_html=True,
        )
        created_at = user["created_at"] or "Not available"
        col1, col2, col3 = st.columns(3)
        with col1:
            render_metric_card("Account Status", "Active", "Authentication account")
        with col2:
            render_metric_card("Assigned Role", str(user["role"]).title(), "Current access level")
        with col3:
            render_metric_card("Member Since", str(created_at)[:10], "Account creation date")
        return

    conn = get_connection()
    try:
        total_users = conn.execute("SELECT COUNT(*) FROM users").fetchone()[0]
        total_admins = conn.execute("SELECT COUNT(*) FROM users WHERE lower(role) = 'admin'").fetchone()[0]
        cutoff = (dt.datetime.now(dt.timezone.utc).date() - dt.timedelta(days=29)).isoformat()
        daily_rows = conn.execute(
            """
            SELECT substr(created_at, 1, 10) AS day, COUNT(*) AS total
            FROM users
            WHERE substr(created_at, 1, 10) >= ?
            GROUP BY substr(created_at, 1, 10)
            ORDER BY day
            """,
            (cutoff,),
        ).fetchall()
    finally:
        conn.close()

    col1, col2, col3 = st.columns(3)
    with col1:
        render_metric_card("Total Accounts", total_users, "All registered accounts")
    with col2:
        render_metric_card("Administrators", total_admins, "Accounts with admin access")
    with col3:
        render_metric_card("Regular Users", total_users - total_admins, "Standard platform accounts")

    st.markdown("### Account Registrations — Last 30 Days")
    days = [cutoff and (dt.date.fromisoformat(cutoff) + dt.timedelta(days=index)).isoformat() for index in range(30)]
    totals_by_day = {row["day"]: row["total"] for row in daily_rows}
    chart_df = pd.DataFrame(
        {"Date": days, "Registrations": [totals_by_day.get(day, 0) for day in days]}
    ).set_index("Date")
    st.bar_chart(chart_df, use_container_width=True)

    st.markdown("### Account Distribution")
    distribution_df = pd.DataFrame(
        {"Accounts": [total_users - total_admins, total_admins]},
        index=["Users", "Administrators"],
    )
    st.bar_chart(distribution_df, use_container_width=True)
    st.caption("These analytics are based on registered account data. Freight and quotation analytics are not available in this milestone.")


def render_dashboard():
    token = st.session_state.token
    payload = verify_jwt(token)
    if not payload:
        st.session_state.token = None
        st.session_state.admin_logged_in = False
        st.session_state.page = "Login"
        st.warning("Your session has expired. Please sign in again.")
        st.rerun()

    email = payload.get("email", "").strip().lower()
    user = get_user_by_email(email)
    if not user:
        st.session_state.token = None
        st.session_state.admin_logged_in = False
        st.session_state.page = "Login"
        st.error("Your account could not be found. Please sign in again.")
        st.rerun()

    if user["permanently_locked"]:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.error("This account has been permanently locked. Contact an administrator to unlock it.")
        st.rerun()

    remaining = get_lock_remaining(user["locked_until"])
    if remaining > 0:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.error(f"This account is temporarily locked. Try again in {format_duration(remaining)}.")
        st.rerun()

    is_admin = str(user["role"]).strip().lower() == "admin"
    st.sidebar.markdown(
        """
        <div style="padding:14px 8px 18px;">
          <div style="font-size:21px;font-weight:800;color:#2458B8;">FreightQuote <span style="color:#3478F6;">AI</span></div>
          <div style="color:#70809A;font-size:12px;margin-top:5px;">Your freight workspace</div>
        </div>
        """,
        unsafe_allow_html=True,
    )
    st.sidebar.markdown("---")

    with st.sidebar:
        selected = option_menu(
            menu_title="Navigation",
            options=["Dashboard", "Analytics", "Logout"],
            icons=["speedometer2", "bar-chart-line", "box-arrow-right"],
            default_index=0,
            styles={
                "container": {"padding": "5px", "background-color": "transparent"},
                "icon": {"color": "#3478F6", "font-size": "17px"},
                "nav-link": {"font-size": "14px", "text-align": "left", "margin": "5px 0", "border-radius": "10px", "color": "#263858"},
                "nav-link-selected": {"background-color": "#3478F6", "color": "#FFFFFF", "font-weight": "600"},
            },
        )
        st.markdown("---")
        st.caption(f"Signed in as: {user['username']}")
        st.caption("Role: Administrator" if is_admin else "Role: User")

    if selected == "Logout":
        st.session_state.token = None
        st.session_state.admin_logged_in = False
        clear_reset()
        st.session_state.page = "Login"
        st.rerun()
    elif selected == "Analytics":
        render_analytics(user)
    elif is_admin:
        render_admin_dashboard(user)
    else:
        render_user_dashboard(user)


# -----------------------------------------------------------------------------
# Application entry point
# -----------------------------------------------------------------------------
def main():
    if st.session_state.token:
        render_dashboard()
        return

    current_page = st.session_state.page
    if current_page == "Login":
        render_login_page()
    elif current_page == "Signup":
        render_signup_page()
    elif current_page == "Forgot":
        render_forgot_page()
    elif current_page == "ForgotSecurity":
        render_security_reset_page()
    elif current_page == "ForgotOTP":
        render_otp_reset_page()
    else:
        st.session_state.page = "Login"
        render_login_page()


if __name__ == "__main__":
    main()


Writing app.py


In [5]:
from google.colab import userdata
from pyngrok import ngrok
from pathlib import Path
import os
import sys
import time
import socket
import subprocess

# Load required secrets from Colab's Secrets panel.
required = ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD"]
missing = []
for key in required:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        missing.append(key)
    else:
        os.environ[key] = value

# ADMIN_EMAIL is optional. Set it to the email of the account that should be admin.
try:
    admin_email = userdata.get("ADMIN_EMAIL")
    if admin_email:
        os.environ["ADMIN_EMAIL"] = admin_email.strip().lower()
    else:
        os.environ.pop("ADMIN_EMAIL", None)
except Exception:
    os.environ.pop("ADMIN_EMAIL", None)

if missing:
    raise RuntimeError(
        "Add these keys in Google Colab Secrets and enable notebook access: "
        + ", ".join(missing)
    )


app_path = Path("/content/app.py")
if not app_path.is_file():
    raise FileNotFoundError("Run the 'Write complete app.py' cell before launching the app.")

# Stop an earlier process tracked by this notebook, if any.
old_process = globals().get("_FREIGHTQUOTE_STREAMLIT_PROCESS")
if old_process is not None and old_process.poll() is None:
    old_process.terminate()
    try:
        old_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_process.kill()
        old_process.wait()

# Clear a stale server on port 8501 and close old ngrok tunnels.
subprocess.run(
    ["bash", "-lc", "fuser -k 8501/tcp >/dev/null 2>&1 || true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False,
)
try:
    ngrok.kill()
except Exception:
    pass

# Allow the previous ngrok process to stop.
time.sleep(2)

# Configure the ngrok authentication token.
# Configure ngrok only if an authtoken is not already saved.
ngrok_config = Path.home() / ".config" / "ngrok" / "ngrok.yml"

config_text = (
    ngrok_config.read_text(encoding="utf-8")
    if ngrok_config.exists()
    else ""
)

if "authtoken:" not in config_text:
    ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
    print("Ngrok authentication token configured.")
else:
    print("Ngrok authentication token already configured.")

log_path = Path("/content/freightquote_streamlit.log")
log_file = open(log_path, "w", encoding="utf-8")
process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", str(app_path),
        "--server.port=8501", "--server.address=0.0.0.0",
        "--server.headless=true",
    ],
    cwd="/content",
    env=os.environ.copy(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
_FREIGHTQUOTE_STREAMLIT_PROCESS = process

ready = False
deadline = time.monotonic() + 30
while time.monotonic() < deadline:
    if process.poll() is not None:
        break
    try:
        with socket.create_connection(("127.0.0.1", 8501), timeout=1):
            ready = True
            break
    except OSError:
        time.sleep(1)

if not ready:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    log_file.flush()
    log_file.close()
    print("Streamlit did not start. Recent logs:")
    print(log_path.read_text(encoding="utf-8", errors="replace")[-8000:])
    raise RuntimeError("FreightQuote AI failed to start. Review the logs above.")

try:
    tunnel = ngrok.connect(addr=8501, proto="http")
    print("FreightQuote AI is running.")
    print("Public URL:", tunnel.public_url)
    print("Keep this cell running while using the website.")
    print("Stop this cell to end the session.")
    while process.poll() is None:
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopping FreightQuote AI...")
except Exception as exc:
    print("Launcher error:", exc)
    print(log_path.read_text(encoding="utf-8", errors="replace")[-8000:])
finally:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    try:
        ngrok.kill()
    except Exception:
        pass
    if not log_file.closed:
        log_file.close()
    print("Application stopped.")


Ngrok authentication token already configured.
FreightQuote AI is running.
Public URL: https://neuter-slot-poncho.ngrok-free.dev
Keep this cell running while using the website.
Stop this cell to end the session.


Stopping FreightQuote AI...
Application stopped.
